# 🕵️ Dia 19 — O ciclo OSINT, parte 1: planejamento e coleta com qualidade de evidência

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 19* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-19.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Fim da tarde na Vigia. Você tem quarenta abas abertas, um print do Instagram sem data e um domínio que "viu em algum lugar". Rafa aponta o print: *"Que horas você tirou? De qual URL? Se o perfil sumir amanhã, como você prova que ele existiu?"*

Neste notebook você vai:

1. **Jogar "pergunta boa ou pergunta ruim"**: oito perguntas de inteligência.
2. **Escrever o plano e a matriz de fontes** do caso, por formulário.
3. **Fazer uma captura com recibo**: URL, hora UTC, cópia, SHA-256 e a hora do próprio servidor.
4. **Reimplementar o `sha256sum -c`** e descobrir qual captura foi adulterada.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. 🎮 Pergunta boa ou pergunta ruim?

Uma pergunta de inteligência boa é **estreita o bastante para você saber quando ela foi respondida**. Para cada pergunta abaixo, responda **S** (dá para responder e conferir) ou **N** (ampla ou vaga demais).

`MODO_JOGO = False` mostra só o gabarito.

In [ ]:
# Cada item: (pergunta, é_respondível?, comentário do Rafa)
PERGUNTAS = [
    ("Qual é a presença on-line da Cooperativa Serra Azul?",
     False, "Não tem fim. Quando você para? Troque por 'quais domínios', 'quais perfis'."),
    ("Quais domínios imitação da Serra Azul ganharam certificado TLS entre 1º e 7/out/2026?",
     True, "Tem objeto, fonte provável (crt.sh) e janela de tempo. Dá para conferir."),
    ("Quem é o Pescador?",
     False, "Mira uma pessoa e não é trabalho da Vigia. Atribuição é da polícia (Dia 01)."),
    ("Quais hosts resolviam para 203.0.113.77 em 6/out/2026, segundo DNS passivo?",
     True, "Objeto, data e fonte. Respondível e verificável por outra pessoa."),
    ("O golpe é grande?",
     False, "'Grande' em quê? Vítimas, domínios, dinheiro? Escreva a métrica."),
    ("Quais perfis públicos com 'Serra Azul' no nome NÃO estão linkados na página Canais oficiais?",
     True, "Tem régua (a página oficial) e resultado esperado (uma lista)."),
    ("Descubra tudo sobre o servidor do kit.",
     False, "'Tudo' convida a sair do passivo. O painel está fora do escopo."),
    ("A página serraazul-pix.example/regularizar mudou entre o disparo e hoje?",
     True, "Duas datas, uma URL, uma fonte (Wayback CDX, Dia 26)."),
]
print(f"{len(PERGUNTAS)} perguntas carregadas.")

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito

acertos = 0                                   # contador de respostas certas
for i, (pergunta, boa, comentario) in enumerate(PERGUNTAS, start=1):
    print(f"\n[{i}/{len(PERGUNTAS)}] {pergunta}")
    if MODO_JOGO:
        resp = ""
        while resp not in ("S", "N"):         # insiste até vir S ou N
            resp = input("   Respondível e conferível? (S/N): ").strip().upper()
        certo = (resp == "S") == boa           # compara a sua resposta com o gabarito
        acertos += certo                       # True vale 1, False vale 0
        print("   ✅ Isso." if certo else "   ❌ Olha de novo.")
    print(f"   Gabarito: {'boa' if boa else 'ruim'} — {comentario}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(PERGUNTAS)}  (Checkpoint BR: pelo menos 6)")

## 2. 📝 O plano e a matriz de fontes

Preencha o formulário (no Colab os campos aparecem como caixas). A célula gera `notas/ISCA-recon-log.md` no formato da [ficha pt-BR](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/fichas/registro-recon-osint.md), com a matriz de fontes e uma checagem automática: **toda linha da matriz precisa ser passiva**.

In [ ]:
#@title Plano do ciclo OSINT { display-mode: "form" }
pergunta = "Quais domínios, hosts e perfis públicos estão ligados à campanha do 'Pix bloqueado' que se passa pela Serra Azul, quando cada um apareceu e quais compartilham infraestrutura?"  #@param {type:"string"}
registro_autorizacao = "notas/registro-de-autorizacao.md (SHA-256 do Dia 01)"  #@param {type:"string"}
base_legal = "Art. 7º, IX (legítimo interesse da cooperativa na prevenção à fraude)"  #@param {type:"string"}
fora_1 = "Painel do IscaKit e qualquer página atrás de login"  #@param {type:"string"}
fora_2 = "Dados de cooperados capturados pelo kit"  #@param {type:"string"}
fora_3 = "Qualquer pessoa física, inclusive quem esteja por trás do apelido 'Pescador'"  #@param {type:"string"}
fora_4 = "Varredura ativa de 203.0.113.77; contato com perfis falsos"  #@param {type:"string"}
parada = "Cada subpergunta tem uma fonte coletada ou 'tentada, sem resultado'; paro e escalo se aparecer credencial ou dado de cooperado"  #@param {type:"string"}

from pathlib import Path

# A matriz: (subpergunta, fontes prováveis, resultado esperado, é passiva?)
MATRIZ = [
    ("Quais domínios imitação existem além de serraazul-pix.example?",
     "crt.sh, SpiderFoot (Dia 20), operadores de busca (Dia 23)", "Lista de domínios", True),
    ("Quando cada um foi registrado e ganhou certificado?",
     "RDAP (Dia 29), crt.sh", "Datas", True),
    ("Quais compartilham infraestrutura com 203.0.113.77?",
     "DNS A/MX/NS, Maltego (Dia 21)", "Sobreposição de IP, NS, AS", True),
    ("A página falsa mudou desde o disparo?",
     "Wayback CDX (Dia 26), captura própria", "Versões e janelas de mudança", True),
    ("Quais perfis públicos usam o nome da cooperativa?",
     "Busca nas plataformas (Dia 25), página Canais oficiais", "Oficiais x não linkados", True),
]

# Checagem automática: o laboratório só aceita linhas passivas.
ativas = [linha[0] for linha in MATRIZ if not linha[3]]
assert not ativas, f"Linhas não passivas na matriz: {ativas}"

# Checagem do original: pelo menos três itens fora do escopo, nenhum vazio.
fora = [f for f in (fora_1, fora_2, fora_3, fora_4) if f.strip()]
print("✅ Itens fora do escopo:", len(fora), "(mínimo 3)" if len(fora) >= 3 else "⚠️ faltam itens!")

# Monta a tabela em Markdown, uma linha por subpergunta.
tabela = "\n".join(f"| {s} | {f} | {r} | {'Sim' if p else 'NÃO'} |" for s, f, r, p in MATRIZ)
texto = f"""# Registro de reconhecimento OSINT — ISCA-P2

## Planejamento
- Pergunta: {pergunta}
- Registro de autorização: {registro_autorizacao}
- Base legal LGPD: {base_legal}
- Fora do escopo:
""" + "".join(f"  - {f}\n" for f in fora) + f"""- Condição de parada: {parada}

### Matriz de fontes
| Subpergunta | Fontes prováveis | Resultado esperado | Passiva? |
|---|---|---|---|
{tabela}

## Registro de coleta
| # | Dado | Fonte | Data e hora de acesso (UTC) | SHA-256 | Observações |
|---|---|---|---|---|---|
"""
Path("notas").mkdir(exist_ok=True)                       # pasta de notas do aluno
Path("notas/ISCA-recon-log.md").write_text(texto, encoding="utf-8")
print(texto)
print("💾 Salvo em notas/ISCA-recon-log.md")

## 3. 📸 Captura com recibo

Os quatro itens de uma captura com qualidade de evidência: **URL exata, hora UTC, cópia, SHA-256**. Aqui somamos uma quinta testemunha que você não controla: o cabeçalho `Date` que o servidor mandou.

A célula tenta `https://example.com/` (domínio reservado pela IANA justamente para exemplos, ou seja, pode). Sem internet, ela usa uma amostra embutida e diz isso no registro: **o registro nunca finge**.

In [ ]:
import hashlib                                  # SHA-256
import urllib.request                           # HTTP sem instalar nada
from datetime import datetime, timezone         # hora em UTC
from pathlib import Path

def capturar(url, pasta="captures"):
    """Baixa a URL, salva a cópia e devolve o 'recibo' da captura."""
    Path(pasta).mkdir(exist_ok=True)
    hora_utc = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")  # SUA hora
    try:
        pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (estudo OSINT)"})
        with urllib.request.urlopen(pedido, timeout=8) as r:
            corpo = r.read()                    # os bytes exatos que chegaram
            data_servidor = r.headers.get("Date", "(sem cabeçalho Date)")
            origem = "rede"
    except Exception as erro:                   # sem rede? plano B, declarado
        corpo = b"<html><body>AMOSTRA OFFLINE de example.com</body></html>"
        data_servidor = f"(offline: {type(erro).__name__})"
        origem = "amostra embutida"
    nome = url.split("//", 1)[-1].replace("/", "_").strip("_") or "pagina"
    arquivo = Path(pasta) / f"{nome}_{hora_utc.replace(':', '')}.html"
    arquivo.write_bytes(corpo)                  # grava a cópia ANTES de calcular o hash
    sha = hashlib.sha256(arquivo.read_bytes()).hexdigest()   # hash do que está no disco
    return {"url": url, "hora_utc": hora_utc, "date_servidor": data_servidor,
            "arquivo": str(arquivo), "sha256": sha, "origem": origem}

recibo = capturar("https://example.com/")
for chave, valor in recibo.items():
    print(f"{chave:>14}: {valor}")

In [ ]:
import json
import urllib.parse

def wayback_mais_proximo(url):
    """Pergunta à API de disponibilidade da Wayback qual o snapshot mais próximo."""
    api = "https://archive.org/wayback/available?url=" + urllib.parse.quote(url, safe="")
    try:
        with urllib.request.urlopen(api, timeout=8) as r:
            snap = json.load(r).get("archived_snapshots", {}).get("closest")
        return snap["url"] if snap else "sem snapshot (peça um em web.archive.org/save)"
    except Exception as erro:
        # Plano B: não inventamos URL. Registramos a falha, como o original manda.
        return f"consulta à Wayback falhou ({type(erro).__name__}); anotar na linha"

recibo["wayback"] = wayback_mais_proximo(recibo["url"])
print("wayback:", recibo["wayback"])

# A linha do registro de coleta, pronta para colar.
linha = (f"| 1 | Captura de {recibo['url']} | {recibo['url']} | {recibo['hora_utc']} | "
         f"`{recibo['sha256'][:12]}...` | Date do servidor: {recibo['date_servidor']}; "
         f"Wayback: {recibo['wayback']}; origem: {recibo['origem']} |")
with open("notas/ISCA-recon-log.md", "a", encoding="utf-8") as f:
    f.write(linha + "\n")                     # acrescenta, nunca sobrescreve
print(linha)

> 💡 **Duas horas são melhores que uma.** Se a sua hora UTC e o `Date` do servidor discordarem por minutos, anote. O relógio da sua máquina também é evidência e também pode estar errado.

## 4. 🔏 O `sha256sum -c` feito à mão

O Checkpoint do original pede que, amanhã, `shasum -a 256 -c` imprima `OK` em todas as linhas. Vamos entender o que esse comando faz reimplementando-o.

A célula abaixo recria as **10 capturas sintéticas** do caso (as mesmas de [`caso-continuo/dados/dia-19/`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-19)) e o arquivo de hashes feito na hora da coleta. Uma delas foi alterada **depois**.

In [ ]:
# As 10 capturas sintéticas: (arquivo, URL, hora UTC, conteúdo original)
PAGINAS = [
 [
  "01-serraazul-inicio.html",
  "https://serraazul.example/",
  "2026-10-06T09:02:11Z",
  "<html><title>Cooperativa Serra Azul</title><body><h1>Serra Azul</h1><p>Cooperativa de credito. 40 mil cooperados.</p></body></html>"
 ],
 [
  "02-serraazul-seguranca.html",
  "https://serraazul.example/seguranca",
  "2026-10-06T09:03:40Z",
  "<html><title>Seguranca</title><body><p>A Serra Azul nunca envia SMS com links para desbloqueio de chave Pix.</p></body></html>"
 ],
 [
  "03-serraazul-canais-oficiais.html",
  "https://serraazul.example/canais-oficiais",
  "2026-10-06T09:05:02Z",
  "<html><title>Canais oficiais</title><body><ul><li>Instagram: @serraazul</li><li>Facebook: /serraazulcoop</li><li>YouTube: @serraazulcoop</li><li>LinkedIn: cooperativa-serra-azul</li></ul><p>Nao temos outro perfil no Instagram.</p></body></html>"
 ],
 [
  "04-serraazul-privacidade.html",
  "https://serraazul.example/privacidade",
  "2026-10-06T09:06:31Z",
  "<html><title>Privacidade</title><body><p>Controladora: Cooperativa Serra Azul. Operadores: provedor de e-mail e de hospedagem.</p></body></html>"
 ],
 [
  "05-serraazul-contato.html",
  "https://serraazul.example/contato",
  "2026-10-06T09:07:58Z",
  "<html><title>Contato</title><body><p>Ouvidoria e agencias. Telefone ficticio +55 00 00000-0099.</p></body></html>"
 ],
 [
  "06-falso-inicio.html",
  "https://serraazul-pix.example/",
  "2026-10-06T09:12:20Z",
  "<html><title>Serra Azul - Regularize sua chave Pix</title><body><script>gtag('config','G-ISCA7K2Q9P')</script><p>Sua chave Pix foi bloqueada por seguranca.</p><!-- IscaKit v2 --></body></html>"
 ],
 [
  "07-falso-formulario.html",
  "https://serraazul-pix.example/regularizar",
  "2026-10-06T09:13:05Z",
  "<html><title>Regularizar</title><body><form><input name=agencia><input name=conta><input name=senha></form></body></html>"
 ],
 [
  "08-falso-desbloqueada.html",
  "https://serraazul-pix.example/ok",
  "2026-10-06T09:14:10Z",
  "<html><title>Chave desbloqueada</title><body><p>Pronto! Sua chave foi regularizada.</p></body></html>"
 ],
 [
  "09-falso-erro-404.html",
  "https://serraazul-pix.example/qualquer",
  "2026-10-06T09:14:52Z",
  "<html><title>404</title><body><p>Pagina nao encontrada.</p></body></html>"
 ],
 [
  "10-falso-erro-sessao.html",
  "https://serraazul-pix.example/sessao-expirada",
  "2026-10-06T09:15:30Z",
  "<html><title>Sessao expirada</title><body><p>Sua sessao expirou. Volte ao inicio.</p></body></html>"
 ]
]

from pathlib import Path
import hashlib

Path("capturas").mkdir(exist_ok=True)
linhas = []
for nome, url, quando, conteudo in PAGINAS:
    dados = conteudo.encode("utf-8")
    linhas.append(f"{hashlib.sha256(dados).hexdigest()}  capturas/{nome}")  # hash na hora da coleta
    Path("capturas", nome).write_bytes(dados)
Path("hashes-20261006T091600Z.txt").write_text("\n".join(linhas) + "\n", encoding="utf-8")

# Alguém "melhora" uma captura depois do hash (é o que vamos detectar).
alvo = Path("capturas/07-falso-formulario.html")
alvo.write_text(alvo.read_text(encoding="utf-8").replace("</form>", "<input name=cpf></form>"), encoding="utf-8")
print("10 capturas e 1 arquivo de hashes criados. Uma foi mexida.")

In [ ]:
def verificar(arquivo_hashes):
    """Faz o que 'sha256sum -c' faz: recalcula cada hash e compara com o anotado."""
    falhas = []
    for linha in Path(arquivo_hashes).read_text(encoding="utf-8").splitlines():
        if not linha.strip():
            continue                                  # pula linhas vazias
        esperado, caminho = linha.split("  ", 1)      # formato: <hash><2 espaços><arquivo>
        try:
            atual = hashlib.sha256(Path(caminho).read_bytes()).hexdigest()
        except FileNotFoundError:
            print(f"{caminho}: FALHOU (arquivo sumiu)")
            falhas.append(caminho)
            continue
        ok = (atual == esperado)                      # um único bit diferente muda tudo
        print(f"{caminho}: {'OK' if ok else 'FALHOU'}")
        if not ok:
            falhas.append(caminho)
    return falhas

MODO_JOGO = True
if MODO_JOGO:
    for i, (nome, *_r) in enumerate(PAGINAS, start=1):
        print(f"  {i:2d}. {nome}")
    palpite = ""
    while palpite not in [str(n) for n in range(1, 11)]:
        palpite = input("Antes de verificar: qual número você acha que foi adulterado? (1-10): ").strip()
    print(f"Seu palpite: {PAGINAS[int(palpite) - 1][0]}\n")

falhas = verificar("hashes-20261006T091600Z.txt")
print(f"\n{len(falhas)} falha(s): {falhas}")

In [ ]:
# Por que justo o formulário? Compare o original com o que está no disco.
import difflib
original = dict((n, c) for n, _u, _q, c in PAGINAS)
for caminho in falhas:
    nome = Path(caminho).name
    antes = original[nome].replace("><", ">\n<").splitlines()          # quebra para o diff ficar legível
    depois = Path(caminho).read_text(encoding="utf-8").replace("><", ">\n<").splitlines()
    for d in difflib.unified_diff(antes, depois, "na coleta", "no disco hoje", lineterm=""):
        print(d)
print("\nAlguém acrescentou um campo de CPF ao formulário depois da coleta.")
print("Sem o hash, essa versão 'mais grave' poderia ir parar no relatório como se fosse a original.")

In [ ]:
# Baixa as notas no Colab (fora dele, ficam na pasta notas/).
try:
    from google.colab import files
    files.download("notas/ISCA-recon-log.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/ISCA-recon-log.md)")

### 🧠 Camada 8 — leia antes de fechar

A adulteração acima "piorava" o golpe (um campo de CPF a mais). É o tipo de mudança que ninguém questiona, porque confirma o que todo mundo já acha. O hash não tem opinião: ele só diz que o arquivo mudou. Esse é o valor de travar a evidência **antes** de ter uma história para contar.

## ✅ Antes de fechar o Dia 19

- [ ] Quiz com pelo menos 6/8
- [ ] `notas/ISCA-recon-log.md` com pergunta, autorização, base legal, 3+ itens fora do escopo, condição de parada e matriz
- [ ] Uma captura com recibo (URL, hora UTC, SHA-256, `Date` do servidor, Wayback ou falha anotada)
- [ ] Você achou o arquivo adulterado **e** sabe explicar por que o `-c` pegou
- [ ] Rodou de verdade: `cd caso-continuo/dados/dia-19 && sha256sum -c hashes-*.txt`

**Amanhã (Dia 20):** o SpiderFoot vai devolver centenas de dados sobre `serraazul-pix.example`. A maioria é ruído de hospedagem compartilhada. Rafa vai perguntar quais você jogou fora, quantos e por quê.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>